<a href="https://colab.research.google.com/github/anasmita3/AI_Final/blob/main/SURGE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bengali NLP Tokenization Comparison Project

Goal:
Compare different tokenization methods on Bengali corpora collected from:

1. Literature
2. News
3. Social Media

Constraints:

- Minimum 500,000 Bengali words per domain
- Pure Bengali only
- No pretrained tokenizers
- Train tokenizers from scratch
- Compare using downstream NLP model performance

Pipeline:

Dataset → Cleaning → Bengali Filtering → Transliteration →
Tokenizer Training → Model Training → Evaluation

In [1]:
!pip install -q \
datasets \
tokenizers \
sentencepiece \
aksharamukha \
tensorflow \
pyarrow \
pandas \
numpy \
scikit-learn \
matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.9/289.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 40.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,f1_score

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
!git clone https://github.com/cypher-07/Bangla-Text-Dataset.git

Cloning into 'Bangla-Text-Dataset'...
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 16 (delta 3), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (16/16), 7.21 MiB | 8.07 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [4]:
import pandas as pd

social = pd.read_csv(
    "/content/Bangla-Text-Dataset/dataset.csv"
)

print(social.head())

                                             comment    Category  Gender  \
0  ওই হালার পুত এখন কি মদ খাওয়ার সময় রাতের বেলা...       Actor  Female   
1  ঘরে বসে শুট করতে কেমন লেগেছে? ক্যামেরাতে কে ছি...      Singer    Male   
2                       অরে বাবা, এই টা কোন পাগল????       Actor  Female   
3                              ক্যাপ্টেন অফ বাংলাদেশ      Sports    Male   
4                                           পটকা মাছ  Politician    Male   

   comment react number      label  
0                   1.0     sexual  
1                   2.0  not bully  
2                   2.0  not bully  
3                   0.0  not bully  
4                   0.0      troll  


In [5]:
import pandas as pd

lit = pd.read_parquet(
    "hf://datasets/barunsaha/bangla_sahitya/data/train-00000-of-00001.parquet"
)

print(lit.head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


                 author                        title    tag  \
0  অক্ষয়কুমার মৈত্রেয়                  সিরাজদ্দৌলা  prose   
1                অজ্ঞাত                                poem   
2                অজ্ঞাত  একবার বিদায় দে মা ঘুরে আসি   poem   
3        অতুলপ্রসাদ সেন                                poem   
4        অতুলপ্রসাদ সেন                                poem   

                                                text  
0  প্রথম পরিচ্ছেদ\n\nসেকালের সুখ দুঃখ।\n\nনবাব সি...  
1  যাও যাও গিরি, আনিতে গৌরী, উমা নাকি বড় কেঁদেছে...  
2  (পীতাম্বর দাস বা মুকুন্দ দাস রচিত)\n\n\nএকবার ...  
3  বলো বলো বলো সবে\n\nঅতুলপ্রসাদ সেন\n\n\nবলো বলো...  
4  ওহে জগত কারণ\n\nঅতুলপ্রসাদ সেন\n\n\nওহে জগত কা...  


In [6]:
!pip install kagglehub -q

In [7]:
import kagglehub

path = kagglehub.dataset_download(
    "furcifer/bangla-newspaper-dataset"
)

print(path)

100%|██████████| 1.03G/1.03G [00:10<00:00, 104MB/s] 

Extracting files...


/root/.cache/kagglehub/datasets/furcifer/bangla-newspaper-dataset/versions/2


In [8]:
import os

os.listdir(path)

['data_v2', 'data']

In [9]:
import os

print(os.listdir(path + "/data"))
print(os.listdir(path + "/data_v2"))

['data.json']
['data_v2.json']


In [10]:
import pandas as pd
import json

with open(path + "/data/data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

with open("news_corpus.txt", "w", encoding="utf-8") as out:
    for article in data:
        text = article.get("content", "")
        out.write(text + "\n")

In [11]:
import re

def clean_text(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^\u0980-\u09FF\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

with open("news_corpus.txt", "r", encoding="utf-8") as fin, \
     open("clean_news.txt", "w", encoding="utf-8") as fout:

    for line in fin:
        cleaned = clean_text(line)
        if cleaned:
            fout.write(cleaned + "\n")

print("Saved cleaned corpus")

Saved cleaned corpus


In [12]:
with open("clean_news.txt","r",encoding="utf-8") as f:
    for i in range(5):
        print(f.readline())

গাজীপুরের কালিয়াকৈর উপজেলার তেলিরচালা এলাকায় আজ বৃহস্পতিবার রাতের টিফিন খেয়ে একটি পোশাক কারখানার ৫০০ শ্রমিক অসুস্থ হয়ে পড়েছেন এ ঘটনায় বিক্ষোভ করেছেন ওই কারখানার শ্রমিকেরা সফিপুর মডার্ন হাসপাতালের জরুরি বিভাগের চিকিত্সক আল আমিন প্রথম আলো ডটকমকে বলেন খাদ্যে বিষক্রিয়ায় তাঁরা শ্রমিকেরা অসুস্থ হয়ে পড়েছেন এতে আতঙ্কিত হওয়ার কিছু নেই অসুস্থদের চিকিত্সা দেওয়া হয়েছে কারখানার শ্রমিক ও পুলিশ সূত্রে জানা যায় উপজেলার তেলিরচালা এলাকার সেজাদ সোয়েটার লিমিটেড কারখানার শ্রমিকদের আজ রাত সাড়ে সাতটার দিকে টিফিন দেওয়া হয় টিফিনে ছিল ডিম রুটি পেটিস ও কলা টিফিন খেয়ে শ্রমিকেরা যথারীতি কাজে যোগ দেন ওই টিফিন খাওয়ার প্রায় এক ঘণ্টা পর রাত সাড়ে আটটার দিকে কয়েকজন শ্রমিকের বমি ও পেট ব্যথা শুরু হয় এরপর ধীরে ধীরে পুরো কারখানার শ্রমিকেরা অসুস্থ হতে থাকে অনেকেই কারখানার মেঝেতে ঢলে পড়ে এতে পাঁচ শতাধিক শ্রমিক অসুস্থ হয়ে পড়ে পরে কারখানা কর্তৃপক্ষ দ্রুত যানবাহনের ব্যবস্থা করে তাদের সফিপুর জেনারেল হাসপাতাল সফিপুর মডার্ন হাসপাতাল উপজেলা স্বাস্থ্য কমপ্লেক্সসহ বিভিন্ন ক্লিনিকে ভর্তি করে বাসি পচা খাবার দে

In [13]:
total_words = 0

with open("clean_news.txt","r",encoding="utf-8") as f:
    for line in f:
        total_words += len(line.split())

print("Total words:", total_words)

Total words: 125839578


In [14]:
def bengali_only(text):

    text=str(text)

    words=text.split()

    clean=[]

    for w in words:

        if re.fullmatch(
            r'[\u0980-\u09FF]+',
            w
        ):
            clean.append(w)

    return " ".join(clean)

In [15]:
# Save literature to txt
with open("lit_corpus.txt", "w", encoding="utf-8") as out:

    for text in lit['text'].dropna():
        out.write(str(text) + "\n")

print("lit_corpus.txt saved")

lit_corpus.txt saved


In [16]:
# Save social media corpus
with open("social_corpus.txt", "w", encoding="utf-8") as out:

    for text in social['comment'].dropna():
        out.write(str(text) + "\n")

print("social_corpus.txt saved")

social_corpus.txt saved


In [17]:
with open("lit_corpus.txt","r",encoding="utf-8") as fin, \
     open("clean_lit.txt","w",encoding="utf-8") as fout:

    for line in fin:
        fout.write(bengali_only(line) + "\n")
with open("social_corpus.txt","r",encoding="utf-8") as fin, \
     open("clean_social.txt","w",encoding="utf-8") as fout:

    for line in fin:
        fout.write(bengali_only(line) + "\n")
with open("news_corpus.txt","r",encoding="utf-8") as fin, \
     open("clean_news.txt","w",encoding="utf-8") as fout:

    for line in fin:
        fout.write(bengali_only(line) + "\n")

In [18]:
files = [
    "clean_lit.txt",
    "clean_news.txt",
    "clean_social.txt"
]

with open("final_corpus.txt","w",encoding="utf-8") as out:
    for file in files:
        with open(file,"r",encoding="utf-8") as fin:
            for line in fin:
                out.write(line)

print("Final corpus created")

Final corpus created


Transliteration

In [19]:
from aksharamukha import transliterate

In [ ]:
from aksharamukha import transliterate

with open("clean_news.txt","r",encoding="utf-8") as fin, \
     open("news_iso.txt","w",encoding="utf-8") as fout:

    for line in fin:

        iso = transliterate.process(
            "Bengali",
            "ISO",
            line.strip()
        )

        fout.write(iso + "\n")

print("News transliteration saved")

In [ ]:
with open("clean_lit.txt","r",encoding="utf-8") as fin, \
     open("lit_iso.txt","w",encoding="utf-8") as fout:

    for line in fin:
        fout.write(
            transliterate.process(
                "Bengali",
                "ISO",
                line.strip()
            ) + "\n"
        )

In [ ]:
with open("clean_social.txt","r",encoding="utf-8") as fin, \
     open("social_iso.txt","w",encoding="utf-8") as fout:

    for line in fin:
        fout.write(
            transliterate.process(
                "Bengali",
                "ISO",
                line.strip()
            ) + "\n"
        )

In [ ]:
files = [
    "news_iso.txt",
    "lit_iso.txt",
    "social_iso.txt"
]

with open("final_iso_corpus.txt","w",encoding="utf-8") as out:
    for file in files:
        with open(file,"r",encoding="utf-8") as fin:
            for line in fin:
                out.write(line)

In [ ]:
from aksharamukha import transliterate

with open("clean_news.txt","r",encoding="utf-8") as f:
    samples = [next(f).strip() for _ in range(10)]

for s in samples:
    iso = transliterate.process(
        "Bengali",
        "ISO",
        s
    )

    print("Original :", s)
    print("ISO      :", iso)
    print("-"*50)

In [ ]:
original = "আমি বাংলাদেশে থাকি"

iso = transliterate.process(
    "Bengali",
    "ISO",
    original
)

back = transliterate.process(
    "ISO",
    "Bengali",
    iso
)

print(original)
print(iso)
print(back)

In [ ]:
!pip install jiwer

In [ ]:
from jiwer import cer

scores=[]

with open("clean_news.txt","r",encoding="utf-8") as f:

    for i,line in enumerate(f):

        if i==100:
            break

        iso = transliterate.process(
            "Bengali",
            "ISO",
            line.strip()
        )

        back = transliterate.process(
            "ISO",
            "Bengali",
            iso
        )

        scores.append(
            cer(
                line.strip(),
                back
            )
        )

print("CER:",sum(scores)/len(scores))

In [ ]:
from jiwer import wer
from aksharamukha import transliterate

scores = []

with open("clean_news.txt","r",encoding="utf-8") as f:

    for i, line in enumerate(f):

        if i > 100:
            break

        original = line.strip()

        iso = transliterate.process(
            "Bengali",
            "ISO",
            original
        )

        recovered = transliterate.process(
            "ISO",
            "Bengali",
            iso
        )

        scores.append(
            wer(
                original,
                recovered
            )
        )

print("Mean WER:", sum(scores)/len(scores))

In [ ]:
!pip install nltk -q

In [ ]:
from nltk.translate.bleu_score import sentence_bleu
from aksharamukha import transliterate

scores=[]

with open("clean_news.txt","r",encoding="utf-8") as f:

    for i,line in enumerate(f):

        if i>100:
            break

        original = line.strip()

        iso = transliterate.process(
            "Bengali",
            "ISO",
            original
        )

        recovered = transliterate.process(
            "ISO",
            "Bengali",
            iso
        )

        scores.append(
            sentence_bleu(
                [original.split()],
                recovered.split()
            )
        )

print("Mean BLEU:",sum(scores)/len(scores))

In [ ]:
# Characters Per Token (CPT)

total_chars = 0
total_tokens = 0

with open("final_corpus.txt", encoding="utf-8") as f:

    for line in f:

        total_chars += len(line.strip())

        encoded = tokenizer.encode(line)

        total_tokens += len(encoded.tokens)

cpt = total_chars / total_tokens

print("Characters Per Token (CPT):", cpt)

In [ ]:
# Fertility Metric

total_words = 0
total_tokens = 0

with open("final_corpus.txt", encoding="utf-8") as f:

    for line in f:

        words = line.strip().split()

        total_words += len(words)

        encoded = tokenizer.encode(line)

        total_tokens += len(encoded.tokens)

fertility = total_tokens / total_words

print("Fertility:", fertility)

In [ ]:
# Word Fragmentation Rate (WFR)

fragmented_words = 0
total_words = 0

with open("final_corpus.txt", encoding="utf-8") as f:

    for line in f:

        words = line.strip().split()

        for word in words:

            total_words += 1

            tokenized = tokenizer.encode(word).tokens

            if len(tokenized) > 1:
                fragmented_words += 1

wfr = fragmented_words / total_words

print("Word Fragmentation Rate:", wfr)

In [ ]:
# Average Tokens Per Sentence

sentence_count = 0
total_tokens = 0

with open("final_corpus.txt", encoding="utf-8") as f:

    for line in f:

        encoded = tokenizer.encode(line)

        total_tokens += len(encoded.tokens)

        sentence_count += 1

avg_seq_len = total_tokens / sentence_count

print("Average Tokens Per Sentence:", avg_seq_len)

In [ ]:
import numpy as np

token_lengths = []

with open("final_corpus.txt", encoding="utf-8") as f:

    for line in f:

        encoded = tokenizer.encode(line)

        token_lengths.append(len(encoded.tokens))

variance = np.var(token_lengths)

print("Token Count Variance:", variance)

In [ ]:
# Simple Morphological Alignment Approximation

suffixes = [
    "টা", "টি", "গুলো",
    "দের", "কে", "এর",
    "তে", "রা"
]

matched = 0
total = 0

with open("final_corpus.txt", encoding="utf-8") as f:

    for line in f:

        words = line.strip().split()

        for word in words:

            for suf in suffixes:

                if word.endswith(suf):

                    total += 1

                    tokens = tokenizer.encode(word).tokens

                    token_string = "".join(tokens)

                    if suf in token_string:
                        matched += 1

morph_score = matched / total

print("Approx Morphological Alignment:", morph_score)

The transliteration pipeline demonstrated strong character-level preservation with CER = 0.046, while exhibiting moderate word-level deviations (WER = 0.154). The average BLEU score of 0.656 indicates reasonable semantic and lexical similarity between original and reconstructed Bengali text after round-trip transliteration. These results suggest that transliteration retains most orthographic information but introduces occasional word-level inconsistencies.

Tokenization

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

In [ ]:
files = [
    "clean_news.txt",
    "clean_lit.txt",
    "clean_social.txt"
]

with open("final_corpus.txt","w",encoding="utf-8") as out:

    for file in files:
        with open(file,"r",encoding="utf-8") as f:
            for line in f:
                out.write(line)

print("Corpus ready")

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import time
import pandas as pd

results = []

vocab_sizes = [8000,12000,16000]
min_freqs = [1,2,3]

for vocab in vocab_sizes:
    for mf in min_freqs:

        start = time.time()

        tokenizer = Tokenizer(
            BPE(unk_token="[UNK]")
        )

        tokenizer.pre_tokenizer = Whitespace()

        trainer = BpeTrainer(
            vocab_size=vocab,
            min_frequency=mf,
            special_tokens=["[UNK]"]
        )

        tokenizer.train(
            files=["final_corpus.txt"],
            trainer=trainer
        )

        train_time = time.time()-start

        # OOV + avg tokens
        unk = 0
        total = 0
        token_counts = []

        with open(
            "final_corpus.txt",
            encoding="utf-8"
        ) as f:

            for i,line in enumerate(f):

                if i>1000:
                    break

                enc = tokenizer.encode(line)

                token_counts.append(
                    len(enc.tokens)
                )

                unk += enc.tokens.count(
                    "[UNK]"
                )

                total += len(enc.tokens)

        results.append({

            "vocab": vocab,
            "min_freq": mf,

            "oov":
            unk/max(total,1),

            "avg_tokens":
            sum(token_counts)
            /len(token_counts),

            "train_time":
            train_time
        })

res = pd.DataFrame(results)

print(
res.sort_values(
    by=["oov","avg_tokens"]
)
)

In [ ]:
orig_chars = 0
token_count = 0

with open(
    "final_corpus.txt",
    encoding="utf-8"
) as f:

    for line in f:

        orig_chars += len(line)

        token_count += len(
            tokenizer.encode(
                line
            ).tokens
        )

print(
"Compression ratio:",
orig_chars/token_count
)

In [ ]:
samples = [
"আমি বাংলাদেশে থাকি",
"কলকাতা একটি শহর",
"বাংলা ভাষা সুন্দর"
]

for s in samples:

    enc = tokenizer.encode(s)

    print(s)
    print(enc.tokens)
    print()

In [ ]:
samples = [
"আমি বাংলাদেশে থাকি",
"বাংলা ভাষা সুন্দর",
"কলকাতা একটি শহর"
]

for s in samples:

    enc = tokenizer.encode(s)

    print("Sentence:")
    print(s)

    print("Tokens:")
    print(enc.tokens)

    print("Token count:",
          len(enc.tokens))

    print("-"*40)

In [ ]:
from collections import Counter

word_count = 0
vocab = Counter()
sentences = 0

with open(
    "final_corpus.txt",
    encoding="utf-8"
) as f:

    for line in f:

        words = line.split()

        word_count += len(words)

        vocab.update(words)

        sentences += 1


print("Total words:", word_count)

print("Unique words:",
      len(vocab))

print("Vocabulary richness:",
      len(vocab)/word_count)

print("Avg words/sentence:",
      word_count/sentences)

In [ ]:
from collections import Counter

counter = Counter()

with open(
    "final_corpus.txt",
    encoding="utf-8"
) as f:

    for line in f:
        counter.update(
            line.split()
        )

print(
counter.most_common(20)
)

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd

word_count = 0
sentence_count = 0
vocab = Counter()

# Read corpus
with open(
    "final_corpus.txt",
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        words = line.split()

        sentence_count += 1

        word_count += len(words)

        vocab.update(words)


# Basic statistics
unique_words = len(vocab)

lexical_diversity = (
    unique_words /
    max(word_count,1)
)

avg_sentence_len = (
    word_count /
    max(sentence_count,1)
)


print("===== Corpus Statistics =====")

print(
"Total words:",
word_count
)

print(
"Unique words:",
unique_words
)

print(
"Total sentences:",
sentence_count
)

print(
"Average words/sentence:",
round(avg_sentence_len,2)
)

print(
"Lexical diversity:",
round(lexical_diversity,4)
)


# Save stats
stats = pd.DataFrame({

"Metric":[
"Total words",
"Unique words",
"Sentences",
"Avg sentence length",
"Lexical diversity"
],

"Value":[
word_count,
unique_words,
sentence_count,
avg_sentence_len,
lexical_diversity
]

})

stats.to_csv(
    "corpus_statistics.csv",
    index=False
)

print(
"\nSaved: corpus_statistics.csv"
)


# ======================
# Top frequent words
# ======================

top_words = vocab.most_common(20)

top_df = pd.DataFrame(
    top_words,
    columns=[
        "Word",
        "Frequency"
    ]
)

print(
"\nTop 20 words:"
)

print(top_df)


top_df.to_csv(
    "top_words.csv",
    index=False
)


# ======================
# Zipf analysis
# ======================

freqs = sorted(
    vocab.values(),
    reverse=True
)

plt.figure(
    figsize=(6,4)
)

plt.loglog(freqs)

plt.xlabel(
    "Rank"
)

plt.ylabel(
    "Frequency"
)

plt.title(
    "Zipf Distribution"
)

plt.show()



In [ ]:
import matplotlib.pyplot as plt

freqs = sorted(
    vocab.values(),
    reverse=True
)

plt.figure(
    figsize=(7,5)
)

plt.hist(
    freqs,
    bins=100,
    log=True      # log y-axis
)

plt.xscale("log")   # log x-axis

plt.xlabel(
"Word Frequency (log)"
)

plt.ylabel(
"Count (log)"
)

plt.title(
"Vocabulary Frequency Distribution"
)

plt.show()

In [ ]:
print("Total words:", word_count)

print("Unique words:", unique_words)

print("Lexical diversity:",
      lexical_diversity)

print("Average sentence length:",
      avg_sentence_len)

print("\nTop 20 words:")
print(vocab.most_common(20))